# Notebook 04: Dimensión de Validez

## Introducción

La segunda dimensión de calidad de datos es la **Validez**: asegurar que los datos cumplan con las reglas de formato, tipo y dominio esperados.

**Duración**: 30 minutos
**Nivel**: Principiante

### Objetivos:

1. Entender qué es validez y por qué importa
2. Validar rangos numéricos
3. Validar dominios (valores permitidos)
4. Validar formatos con regex
5. Generar reportes de validez

## ¿Qué es Validez?

**Validez** mide si los datos cumplen con las reglas de formato, tipo y dominio esperados.

### Pregunta Clave:
> ¿Los valores están dentro de los rangos y formatos permitidos?

### Tipos de Validez:

1. **Validez de Rango**: Valores numéricos dentro de límites
2. **Validez de Dominio**: Valores dentro de un conjunto permitido
3. **Validez de Formato**: Valores que cumplen un patrón (regex)
4. **Validez de Tipo**: Valores del tipo de dato correcto

### Impacto de Negocio:

- **Pérdidas financieras**: Precios negativos, cantidades inválidas
- **Procesos rotos**: Datos inválidos rompen pipelines downstream
- **Análisis incorrectos**: Outliers extremos distorsionan estadísticas
- **Problemas legales**: Datos inválidos pueden violar regulaciones

In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np

print(f"Great Expectations versión: {gx.__version__}")

In [ ]:
# Cargar datos
df = pd.read_csv("../data/ventas_sucias.csv")

print(f"Total de registros: {len(df)}")
print(f"\nPrimeras filas:")
df.head(10)

## 1. Validez de Rango

Validar que valores numéricos estén dentro de rangos lógicos.

In [ ]:
# Análisis de rangos
print("=" * 70)
print("ANÁLISIS DE RANGOS NUMÉRICOS")
print("=" * 70)

print("\n1. PRICE (Precio):")
print(f"   Mínimo: ${df['price'].min():.2f}")
print(f"   Máximo: ${df['price'].max():.2f}")
print(f"   Media: ${df['price'].mean():.2f}")
print(f"   Precios negativos: {(df['price'] < 0).sum()}")
print(f"   Precios > $10,000: {(df['price'] > 10000).sum()}")

print("\n2. QUANTITY (Cantidad):")
print(f"   Mínimo: {df['quantity'].min()}")
print(f"   Máximo: {df['quantity'].max()}")
print(f"   Media: {df['quantity'].mean():.2f}")
print(f"   Cantidades <= 0: {(df['quantity'] <= 0).sum()}")
print(f"   Cantidades > 100: {(df['quantity'] > 100).sum()}")

In [ ]:
# Configurar contexto
context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

# Suite de validez de rango
suite_rango = context.suites.add(gx.ExpectationSuite(name="validez_rango"))

# Validar precio
suite_rango.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0.01,
        max_value=10000,
        meta={
            "dimension": "Validez",
            "tipo": "Rango",
            "regla": "Precio debe estar entre $0.01 y $10,000"
        }
    )
)

# Validar cantidad
suite_rango.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="quantity",
        min_value=1,
        max_value=100,
        meta={
            "dimension": "Validez",
            "tipo": "Rango",
            "regla": "Cantidad debe estar entre 1 y 100"
        }
    )
)

suite_rango.save()

# Validar
val_def_rango = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_rango, name="val_rango")
)

resultado_rango = val_def_rango.run(batch_parameters={"dataframe": df})

print("\n" + "=" * 70)
print("RESULTADO: VALIDEZ DE RANGO")
print("=" * 70)
print(f"\n¿Validación exitosa?: {' SÍ' if resultado_rango.success else ' NO'}")

for result in resultado_rango.results:
    if not result.success:
        column = result.expectation_config.kwargs['column']
        unexpected_count = result.result.get('unexpected_count', 0)
        print(f"\n {column}: {unexpected_count} valores fuera de rango")

## 2. Validez de Dominio

Validar que los valores pertenezcan a un conjunto permitido.

In [ ]:
# Análisis de dominios
print("=" * 70)
print("ANÁLISIS DE DOMINIOS (VALORES PERMITIDOS)")
print("=" * 70)

print("\n1. PRODUCT_CATEGORY:")
print(f"   Valores únicos: {df['product_category'].nunique()}")
print(f"   Distribución:")
print(df['product_category'].value_counts())

# Definir categorías válidas
categorias_validas = ["Electronics", "Clothing", "Home", "Toys"]
categorias_invalidas = df[~df['product_category'].isin(categorias_validas)]['product_category'].unique()
print(f"\n   Categorías inválidas encontradas: {list(categorias_invalidas)}")

In [ ]:
# Suite de validez de dominio
suite_dominio = context.suites.add(gx.ExpectationSuite(name="validez_dominio"))

# Validar categorías
suite_dominio.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="product_category",
        value_set=["Electronics", "Clothing", "Home", "Toys"],
        meta={
            "dimension": "Validez",
            "tipo": "Dominio",
            "regla": "Solo categorías de productos válidas"
        }
    )
)

suite_dominio.save()

val_def_dominio = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_dominio, name="val_dominio")
)

resultado_dominio = val_def_dominio.run(batch_parameters={"dataframe": df})

print("\n" + "=" * 70)
print("RESULTADO: VALIDEZ DE DOMINIO")
print("=" * 70)
print(f"\n¿Validación exitosa?: {' SÍ' if resultado_dominio.success else ' NO'}")

for result in resultado_dominio.results:
    if not result.success:
        unexpected_count = result.result.get('unexpected_count', 0)
        unexpected_values = result.result.get('partial_unexpected_list', [])
        print(f"\n Valores inválidos encontrados: {unexpected_count}")
        print(f"   Ejemplos: {unexpected_values[:5]}")

## 3. Validez de Formato

Validar que los valores cumplan con patrones específicos usando expresiones regulares.

In [ ]:
# Análisis de formatos
print("=" * 70)
print("ANÁLISIS DE FORMATOS")
print("=" * 70)

print("\n1. ORDER_ID (debe ser UUID):")
print(f"   Ejemplos:")
for i, order_id in enumerate(df['order_id'].head(5), 1):
    print(f"   {i}. {order_id}")

# Validar formato UUID
import re
uuid_pattern = r'^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$'
validos = df['order_id'].astype(str).str.match(uuid_pattern, na=False).sum()
print(f"\n   UUIDs válidos: {validos}/{len(df)}")
print(f"   UUIDs inválidos: {len(df) - validos}")

In [ ]:
# Suite de validez de formato
suite_formato = context.suites.add(gx.ExpectationSuite(name="validez_formato"))

# Validar formato UUID
suite_formato.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="order_id",
        regex=r"^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$",
        meta={
            "dimension": "Validez",
            "tipo": "Formato",
            "regla": "Order ID debe ser UUID válido"
        }
    )
)

suite_formato.save()

val_def_formato = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_formato, name="val_formato")
)

resultado_formato = val_def_formato.run(batch_parameters={"dataframe": df})

print("\n" + "=" * 70)
print("RESULTADO: VALIDEZ DE FORMATO")
print("=" * 70)
print(f"\n¿Validación exitosa?: {' SÍ' if resultado_formato.success else ' NO'}")

for result in resultado_formato.results:
    if not result.success:
        unexpected_count = result.result.get('unexpected_count', 0)
        print(f"\n Formatos inválidos: {unexpected_count}")

## 4. Suite Maestra de Validez

Combinemos todas las validaciones de validez en una suite maestra.

In [ ]:
# Suite maestra
suite_maestra = context.suites.add(gx.ExpectationSuite(name="validez_completa"))

# RANGO
suite_maestra.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price", min_value=0.01, max_value=10000,
        meta={"dimension": "Validez", "tipo": "Rango"}
    )
)
suite_maestra.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="quantity", min_value=1, max_value=100,
        meta={"dimension": "Validez", "tipo": "Rango"}
    )
)

# DOMINIO
suite_maestra.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="product_category",
        value_set=["Electronics", "Clothing", "Home", "Toys"],
        meta={"dimension": "Validez", "tipo": "Dominio"}
    )
)

# FORMATO
suite_maestra.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="order_id",
        regex=r"^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$",
        meta={"dimension": "Validez", "tipo": "Formato"}
    )
)

suite_maestra.save()

val_def_maestra = context.validation_definitions.add(
    gx.ValidationDefinition(data=batch_def, suite=suite_maestra, name="val_completa")
)

resultado_completo = val_def_maestra.run(batch_parameters={"dataframe": df})

print("\n" + "=" * 70)
print("REPORTE COMPLETO DE VALIDEZ")
print("=" * 70)
print(f"\nEstado: {' APROBADO' if resultado_completo.success else ' RECHAZADO'}")
print(f"\nTotal expectativas: {len(resultado_completo.results)}")
print(f"Exitosas: {sum(1 for r in resultado_completo.results if r.success)}")
print(f"Fallidas: {sum(1 for r in resultado_completo.results if not r.success)}")
print(f"Tasa de éxito: {(sum(1 for r in resultado_completo.results if r.success) / len(resultado_completo.results) * 100):.1f}%")

##  Ejercicio Práctico

Agrega validaciones de validez para un campo de email (si existiera en el dataset).

### Requisitos:
1. El email debe cumplir con formato válido
2. El dominio debe ser uno de: gmail.com, yahoo.com, hotmail.com, empresa.com

### Pistas:
- Usa `ExpectColumnValuesToMatchRegex` para formato
- Regex de email: `^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$`
- Para dominio, puedes usar regex o crear una columna derivada

In [ ]:
# TU CÓDIGO AQUÍ
# Crea una suite para validar emails

# suite_email = context.suites.add(gx.ExpectationSuite(name="validez_email"))
# ...

pass

## Generar Data Docs

In [ ]:
context.build_data_docs()

print("\n" + "=" * 70)
print(" Data Docs de Validez generados!")
print("=" * 70)
print("\nEn el reporte verás:")
print("  - Validaciones de rango")
print("  - Validaciones de dominio")
print("  - Validaciones de formato")
print("  - Ejemplos de valores inválidos")

context.open_data_docs()

##  Resumen del Notebook

Has aprendido sobre la dimensión de **Validez**:

### Conceptos Clave:

1.  Validez asegura que datos cumplan reglas de formato, tipo y dominio
2.  Tres tipos: Rango, Dominio, Formato
3.  Usa `ExpectColumnValuesToBeBetween` para rangos
4.  Usa `ExpectColumnValuesToBeInSet` para dominios
5.  Usa `ExpectColumnValuesToMatchRegex` para formatos

### Expectativas Aprendidas:

- `ExpectColumnValuesToBeBetween`: Valida rangos numéricos
- `ExpectColumnValuesToBeInSet`: Valida valores permitidos
- `ExpectColumnValuesToMatchRegex`: Valida patrones con regex
